In [1]:
import numpy as np

def prewavelet_transform_matrix(N0: int, J: int):
    """
    Build the sparse (here dense for simplicity) matrix W that maps fine-grid
    nodal coefficients in V_J to pre-wavelet detail coefficients on levels 1..J.
    Size: rows = N0*(2**J - 1), cols = nJ = 2**J * N0 - 1.
    """
    nJ = 2**J * N0 - 1
    rows = []
    meta = []  # (j, ell) -> row index

    # stencil weights
    left  = [(1, 0.9), (2, -0.6), (3, 0.1)]
    mid   = [(-2, 0.1), (-1, -0.6), (0, 1.0), (1, -0.6), (2, 0.1)]
    right = [(-3, 0.1), (-2, -0.6), (-1, 0.9)]

    for j in range(1, J+1):
        s = 2**(J - j)                 # stride on the fine grid
        L = (2**(j-1)) * N0            # number of wavelets on level j

        # l = 1 (left boundary on level j)
        r = np.zeros(nJ)
        for m, w in left:
            k = s*m - 1                # 1-based to 0-based
            r[k] = w
        meta.append((j, 1))
        rows.append(r)

        # 2 <= l <= L-1 (interior)
        for ell in range(2, L):
            r = np.zeros(nJ)
            c = s*(2*ell - 1) - 1      # center (0-based)
            for off, w in mid:
                r[c + off*s] = w
            meta.append((j, ell))
            rows.append(r)

        # l = L (right boundary on level j)
        r = np.zeros(nJ)
        base = s*(2*L - 1) - 1
        for off, w in right:
            r[base + off*s] = w
        meta.append((j, L))
        rows.append(r)

    W = np.vstack(rows)
    return W, meta


In [3]:
N0, J = 5, 3
W, meta = prewavelet_transform_matrix(N0, J)
# u: fine-grid coefficient vector in V_J of length 2**J*N0 - 1
# d = W @ u  gives all pre-wavelet coefficients (levels 1..J)

print(W)


[[ 0.   0.   0.  ...  0.   0.   0. ]
 [ 0.   0.   0.  ...  0.   0.   0. ]
 [ 0.   0.   0.  ...  0.   0.   0. ]
 ...
 [ 0.   0.   0.  ...  0.1  0.   0. ]
 [ 0.   0.   0.  ...  1.  -0.6  0.1]
 [ 0.   0.   0.  ... -0.6  0.9  0. ]]
